In [1]:
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Get the API key
api_key = os.getenv("OPENAI_API_KEY")

print("API Key Loaded:", api_key is not None)

API Key Loaded: True


In [3]:
pip install -U langgraph

  Obtaining dependency information for langgraph from https://files.pythonhosted.org/packages/89/32/772db1b00a9fe42f50320d1aa20caefb76e621eff1f7218b9918093d631d/langgraph-1.2.6-py3-none-any.whl.metadata
  Using cached langgraph-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Obtaining dependency information for langgraph-prebuilt<1.2.0,>=1.1.0 from https://files.pythonhosted.org/packages/e9/43/3fe1a700b8490ed02679cdbbc8c915eb23a092faf496c9c1118abcd10be3/langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
Using cached langgraph-1.2.6-py3-none-any.whl (246 kB)
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install -U langchain
# Requires Python 3.10+


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import langgraph
import langchain
import openai
import fitz

print("Everything is installed successfully! 🚀")

Everything is installed successfully! 🚀


In [9]:
import operator
from typing_extensions import TypedDict, Annotated

In [ ]:
class ResearchState(TypedDict):
    topic: str
    research_plan: list[str]
    research_results: Annotated[list[str], operator.add]
    reviewer_feedback: str
    approved: bool
    report: str
    report_path: str      
    ppt_path: str

In [12]:
def manager_agent(state: ResearchState) -> ResearchState:
    print("\n========== MANAGER AGENT ==========")

    if not state["topic"].strip():
        raise ValueError("Research topic cannot be empty.")

    print(f"Research Topic: {state['topic']}")
    print("Workflow Started Successfully.")

    return state

In [13]:
state = {
    "topic": "Impact of AI in Healthcare",
    "research_plan": [],
    "research_results": [],
    "reviewer_feedback": "",
    "approved": False,
    "report": "",
    "ppt_path": ""
}

manager_agent(state)


========== MANAGER AGENT ==========
Research Topic: Impact of AI in Healthcare
Workflow Started Successfully.


{'topic': 'Impact of AI in Healthcare',
 'research_plan': [],
 'research_results': [],
 'reviewer_feedback': '',
 'approved': False,
 'report': '',
 'ppt_path': ''}

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

c:\Users\Niharika\OneDrive\Desktop\agentic ai project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [ ]:
def planner_agent(state: ResearchState) -> ResearchState:
    print("\n========== PLANNER AGENT ==========")
    prompt = f"""
    You are an expert research planner.
    Create a structured research plan for the topic:
    {state["topic"]}
    Return only a numbered list of research objectives.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    plan = response.content.split("\n")
    state["research_plan"] = plan
    print("Research Plan Created Successfully.")
    return state

In [ ]:
state = planner_agent(state)
print(state["research_plan"])

In [ ]:
from langchain_core.messages import HumanMessage

def research_agent(state: ResearchState) -> ResearchState:

    print("\n========== RESEARCH AGENT ==========")

    for objective in state["research_plan"]:

        # Web Search
        web_results = web_search.invoke(objective)

        # RAG Retrieval
        documents = retriever.invoke(objective)

        rag_context = "\n\n".join(
            doc.page_content
            for doc in documents
        )

        prompt = f"""
        You are an expert research assistant.

        Research Objective:
        {objective}

        Web Search Results:
        {web_results}

        PDF Context:
        {rag_context}

        Combine all the information and generate a detailed research summary.
        """

        response = llm.invoke(
            [HumanMessage(content=prompt)]
        )

        state["research_results"].append(response.content)

    print("Research Completed Successfully.")

    return state

In [ ]:
def reviewer_agent(state: ResearchState) -> ResearchState:
    print("\n========== REVIEWER AGENT ==========")
    prompt = f"""
    You are an expert research reviewer.
    Review the following research.
    {state["research_results"]}
    If the research is complete, respond exactly:
    APPROVED
    Otherwise explain what is missing.
    """
    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )
    feedback = response.content
    state["reviewer_feedback"] = feedback
    state["approved"] = "APPROVED" in feedback.upper()
    print("Review Completed Successfully.")
    return state

In [ ]:
from langchain_core.messages import HumanMessage

def report_agent(state: ResearchState) -> ResearchState:

    print("\n========== REPORT AGENT ==========")

    prompt = f"""
    You are an expert report writer.

    Using the following research, generate a well-structured research report.

    Research:
    {state["research_results"]}
    """

    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    # Store report in state
    state["report"] = response.content

    # Generate Word document
    generate_report(
        state["report"],
        "reports/research_report.docx"
    )
    generate_ppt(
    state["topic"],
    state["report"],
    "presentations/research_presentation.pptx"
)

state["ppt_path"] = "presentations/research_presentation.pptx"

    # Store report path
    state["report_path"] = "reports/research_report.docx"

    print("Report Generated Successfully.")

    return state

In [15]:
from langgraph.graph import StateGraph, START, END

In [16]:
builder = StateGraph(ResearchState)

In [17]:
builder.add_node("manager", manager_agent)
builder.add_node("planner", planner_agent)
builder.add_node("research", research_agent)
builder.add_node("reviewer", reviewer_agent)
builder.add_node("report", report_agent)

NameError: name 'planner_agent' is not defined

In [ ]:
builder.add_edge(START, "manager")
builder.add_edge("manager", "planner")
builder.add_edge("planner", "research")
builder.add_edge("research", "reviewer")

In [ ]:
def review_router(state: ResearchState):
    if state["approved"]:
        return "report"
    return "research"

In [ ]:
builder.add_conditional_edges(
    "reviewer",
    review_router,
    {
        "report": "report",
        "research": "research"
    }
)

In [ ]:
builder.add_edge("report", END)

In [ ]:
graph = builder.compile()

In [ ]:
initial_state = {
    "topic": "Impact of AI in Healthcare",
    "research_plan": [],
    "research_results": [],
    "reviewer_feedback": "",
    "approved": False,
    "report": "",
    "ppt_path": ""
}

result = graph.invoke(initial_state)

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
web_search = TavilySearchResults(
    max_results=5
)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
def load_pdf(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

In [ ]:
documents = load_pdf("sample.pdf")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

In [ ]:
client = QdrantClient(
    path="./vector_db"
)

In [ ]:
vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    path="./vector_db",
    collection_name="research_docs"
)

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
documents = retriever.invoke(
    "Applications of AI in Healthcare"
)

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in documents
)

In [ ]:
from docx import Document

In [ ]:
def generate_report(report: str, filename: str):
    document = Document()
    document.add_heading("Research Report", level=1)
    document.add_paragraph(report)
    document.save(filename)

In [ ]:
from pptx import Presentation

In [ ]:
def generate_ppt(title: str, report: str, filename: str):

    presentation = Presentation()

    # Title Slide
    slide_layout = presentation.slide_layouts[0]
    slide = presentation.slides.add_slide(slide_layout)

    slide.shapes.title.text = title
    slide.placeholders[1].text = "Generated by AI Workflow Simulation"

    # Content Slides
    paragraphs = report.split("\n\n")

    for paragraph in paragraphs:

        slide_layout = presentation.slide_layouts[1]

        slide = presentation.slides.add_slide(slide_layout)

        slide.shapes.title.text = "Research"

        slide.placeholders[1].text = paragraph

    presentation.save(filename)